# Complete Publication Figure

This notebook demonstrates how to create **publication-ready figures** using yplot's SubplotLayout system. We'll build a comprehensive multi-panel figure that combines:

- Multiple plot types (scatter, bar, line, heatmap, distribution)
- Statistical annotations
- Proper subplot labels (A, B, C, ...)
- Consistent styling via rcParams
- Precise dimensions suitable for journal submission

This is the workflow you should use for creating figures for papers and presentations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import yplot
import yplot.plot as yp
from yplot.annotate import add_significance, add_text_box
from yplot.annotate.stats import add_regression_stats

# Apply publication style
yplot.use('publication')

## Generate Realistic Sample Data

Creating data that mimics a typical experimental study.

In [ ]:
np.random.seed(42)

# Treatment comparison data
n_per_group = 30
treatment_df = pd.DataFrame({
    'treatment': (['Control'] * n_per_group + 
                  ['Low Dose'] * n_per_group + 
                  ['High Dose'] * n_per_group),
    'response': np.concatenate([
        np.random.normal(100, 15, n_per_group),  # Control
        np.random.normal(130, 18, n_per_group),  # Low Dose
        np.random.normal(175, 20, n_per_group),  # High Dose
    ])
})

# Time course data
time_points = np.array([0, 1, 2, 4, 8, 12, 24])
time_df = pd.DataFrame({
    'time': np.tile(time_points, 3),
    'concentration': np.concatenate([
        100 * np.exp(-0.1 * time_points) + np.random.normal(0, 5, len(time_points)),
        100 * np.exp(-0.15 * time_points) + np.random.normal(0, 5, len(time_points)),
        100 * np.exp(-0.2 * time_points) + np.random.normal(0, 5, len(time_points)),
    ]),
    'condition': (['Slow'] * len(time_points) + 
                  ['Medium'] * len(time_points) + 
                  ['Fast'] * len(time_points))
})

# Correlation data
n_samples = 50
biomarker1 = np.random.normal(50, 10, n_samples)
biomarker2 = 1.5 * biomarker1 + np.random.normal(0, 12, n_samples)
correlation_df = pd.DataFrame({
    'Biomarker A': biomarker1,
    'Biomarker B': biomarker2
})

# Heatmap data (gene expression)
genes = ['Gene1', 'Gene2', 'Gene3', 'Gene4', 'Gene5', 'Gene6']
samples = ['Ctrl1', 'Ctrl2', 'Trt1', 'Trt2', 'Trt3', 'Trt4']
expression_data = np.random.randn(6, 6)
# Add some structure
expression_data[:3, :2] -= 1  # Control samples, genes 1-3
expression_data[:3, 2:] += 1  # Treatment samples, genes 1-3
expression_df = pd.DataFrame(expression_data, index=genes, columns=samples)

print("Data generated successfully!")
print(f"  - Treatment data: {len(treatment_df)} observations")
print(f"  - Time course data: {len(time_df)} observations")
print(f"  - Correlation data: {len(correlation_df)} pairs")
print(f"  - Expression matrix: {expression_df.shape}")

## Figure 1: Treatment Comparison Study

A 2x2 layout showing treatment effects with statistical annotations.

In [ ]:
# Define layout for a typical journal figure (fits in one column ~3.5" or two columns ~7")
layout = yplot.SubplotLayout(config={
    "fig_size": (7.5, 7),
    "margins": {"left": 0.6, "right": 0.3, "top": 0.3, "bottom": 0.5},
    "row_1": {
        "size": (2.8, 2.8),
        "cols": 2,
        "spacing": {"hspace": 0.9, "wspace": 0.7}
    },
    "row_2": {
        "size": (2.8, 2.8),
        "cols": 2,
        "spacing": {"hspace": 0.9, "wspace": 0.7}
    }
})

fig, axes = yplot.create_figure_with_layout(layout)

# Panel A: Box plot with individual points
ax = axes[0]
yp.box(treatment_df, x='treatment', y='response', showfliers=False, ax=ax)
yp.strip(treatment_df, x='treatment', y='response', size=3, alpha=0.5, ax=ax)
ax.set_ylabel('Response (AU)')
ax.set_xlabel('')
ax.set_ylim(50, 230)

# Add significance brackets
add_significance(ax, 0, 2, 210, pvalue=0.001)
add_significance(ax, 0, 1, 195, pvalue=0.01)

# Panel B: Bar plot with error bars
ax = axes[1]
summary = treatment_df.groupby('treatment').agg(
    mean=('response', 'mean'),
    sem=('response', lambda x: x.std() / np.sqrt(len(x)))
).reset_index()
summary = summary.sort_values('mean')  # Sort by mean

bars = ax.bar(range(3), summary['mean'], yerr=summary['sem'], capsize=3)
ax.set_xticks(range(3))
ax.set_xticklabels(summary['treatment'])
ax.set_ylabel('Mean Response (AU)')
ax.set_xlabel('')
ax.set_ylim(0, 220)

# Add sample size annotation
add_text_box(ax, 0.95, 0.95, f'n = {n_per_group}/group', transform='axes', 
             ha='right', va='top', fontsize=7)

# Panel C: Time course
ax = axes[2]
for condition in ['Slow', 'Medium', 'Fast']:
    subset = time_df[time_df['condition'] == condition]
    ax.plot(subset['time'], subset['concentration'], 'o-', label=condition, markersize=4)
ax.set_xlabel('Time (hours)')
ax.set_ylabel('Concentration (ng/mL)')
ax.legend(fontsize=7, frameon=False)
ax.set_xlim(-1, 25)

# Panel D: Correlation with regression
ax = axes[3]
ax.scatter(correlation_df['Biomarker A'], correlation_df['Biomarker B'], alpha=0.7, s=20)

# Fit and plot regression line
slope, intercept, r, p, _ = stats.linregress(
    correlation_df['Biomarker A'], 
    correlation_df['Biomarker B']
)
x_line = np.array([correlation_df['Biomarker A'].min(), correlation_df['Biomarker A'].max()])
ax.plot(x_line, slope * x_line + intercept, 'r-', lw=1.5)

ax.set_xlabel('Biomarker A')
ax.set_ylabel('Biomarker B')
add_regression_stats(ax, r**2, p, slope, intercept, loc='lower right')

# Add subplot labels
yplot.add_subplot_labels(fig, axes)

plt.show()

# Save figure (uncomment to save)
# fig.savefig('figure1_treatment_study.pdf', dpi=300, bbox_inches='tight')
# fig.savefig('figure1_treatment_study.png', dpi=300, bbox_inches='tight')

## Figure 2: Gene Expression Analysis

A 3-panel figure combining heatmap with summary statistics.

In [ ]:
# Layout: 1 wide panel on top, 2 panels below
layout = yplot.SubplotLayout(config={
    "fig_size": (7.5, 7),
    "margins": {"left": 0.7, "right": 0.8, "top": 0.3, "bottom": 0.5},
    "row_1": {
        "size": (5.5, 2.8),
        "cols": 1,
        "spacing": {"hspace": 0.0, "wspace": 0.7}
    },
    "row_2": {
        "size": (2.8, 2.8),
        "cols": 2,
        "spacing": {"hspace": 0.9, "wspace": 0.7}
    }
})

fig, axes = yplot.create_figure_with_layout(layout)

# Panel A: Heatmap
ax = axes[0]
yp.heatmap(expression_df, cmap='RdBu_r', center=0, annot=True, fmt='.1f', ax=ax)
ax.set_title('Gene Expression (log2 fold change)')

# Panel B: Average expression by condition
ax = axes[1]
ctrl_mean = expression_df.iloc[:, :2].mean(axis=1)
trt_mean = expression_df.iloc[:, 2:].mean(axis=1)

x = np.arange(len(genes))
width = 0.35
ax.bar(x - width/2, ctrl_mean, width, label='Control', color='steelblue')
ax.bar(x + width/2, trt_mean, width, label='Treatment', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(genes, rotation=45, ha='right')
ax.set_ylabel('Mean Expression')
ax.legend(fontsize=7, frameon=False)
ax.axhline(0, color='gray', linestyle='--', lw=0.5)

# Panel C: Volcano-like plot (simulated)
ax = axes[2]
np.random.seed(123)
fold_changes = np.random.randn(100)
pvalues = 10 ** (-np.abs(fold_changes) * np.random.uniform(0.5, 2, 100))
neg_log_p = -np.log10(pvalues)

# Color by significance
significant = (np.abs(fold_changes) > 1) & (neg_log_p > 1.3)
ax.scatter(fold_changes[~significant], neg_log_p[~significant], 
           alpha=0.5, s=15, c='gray', label='NS')
ax.scatter(fold_changes[significant], neg_log_p[significant], 
           alpha=0.7, s=20, c='red', label='Significant')

ax.axhline(1.3, color='gray', linestyle='--', lw=0.5)  # p = 0.05
ax.axvline(-1, color='gray', linestyle='--', lw=0.5)
ax.axvline(1, color='gray', linestyle='--', lw=0.5)

ax.set_xlabel('Log2 Fold Change')
ax.set_ylabel('-Log10(p-value)')
ax.legend(fontsize=7, frameon=False, loc='upper right')

# Add subplot labels
yplot.add_subplot_labels(fig, axes)

plt.show()

## Figure 3: Comprehensive 6-Panel Figure

A complete figure demonstrating various plot types together.

In [ ]:
# 3x2 layout for comprehensive analysis
layout = yplot.SubplotLayout(config={
    "fig_size": (7.5, 9.5),
    "margins": {"left": 0.6, "right": 0.3, "top": 0.3, "bottom": 0.5},
    "row_1": {
        "size": (2.8, 2.5),
        "cols": 2,
        "spacing": {"hspace": 0.9, "wspace": 0.6}
    },
    "row_2": {
        "size": (2.8, 2.5),
        "cols": 2,
        "spacing": {"hspace": 0.9, "wspace": 0.6}
    },
    "row_3": {
        "size": (2.8, 2.5),
        "cols": 2,
        "spacing": {"hspace": 0.9, "wspace": 0.6}
    }
})

fig, axes = yplot.create_figure_with_layout(layout)

# Panel A: Violin plot
ax = axes[0]
yp.violin(treatment_df, x='treatment', y='response', alpha=0.7, ax=ax)
yp.strip(treatment_df, x='treatment', y='response', size=2, alpha=0.5, ax=ax)
ax.set_ylabel('Response (AU)')
ax.set_xlabel('')

# Panel B: Scatter with regression
ax = axes[1]
ax.scatter(correlation_df['Biomarker A'], correlation_df['Biomarker B'], 
           alpha=0.7, s=20, c='steelblue')
slope, intercept, r, p, _ = stats.linregress(
    correlation_df['Biomarker A'], correlation_df['Biomarker B']
)
x_line = np.linspace(25, 75, 100)
ax.plot(x_line, slope * x_line + intercept, 'r-', lw=1.5)
ax.fill_between(x_line, 
                slope * x_line + intercept - 15, 
                slope * x_line + intercept + 15, 
                alpha=0.2, color='red')
ax.set_xlabel('Biomarker A')
ax.set_ylabel('Biomarker B')
add_text_box(ax, 0.05, 0.95, f'R² = {r**2:.3f}\np < 0.001', 
             transform='axes', fontsize=7)

# Panel C: Time course with error bands
ax = axes[2]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for i, condition in enumerate(['Slow', 'Medium', 'Fast']):
    subset = time_df[time_df['condition'] == condition]
    y = subset['concentration'].values
    # Simulated error
    yerr = np.abs(y) * 0.1
    ax.fill_between(subset['time'], y - yerr, y + yerr, alpha=0.2, color=colors[i])
    ax.plot(subset['time'], y, 'o-', label=condition, markersize=4, color=colors[i])
ax.set_xlabel('Time (hours)')
ax.set_ylabel('Concentration (ng/mL)')
ax.legend(fontsize=7, frameon=False)

# Panel D: Bar chart with significance
ax = axes[3]
means = [100, 135, 178]
sems = [5, 7, 8]
bars = ax.bar(['Control', 'Low', 'High'], means, yerr=sems, capsize=3,
              color=['steelblue', 'coral', 'firebrick'])
ax.set_ylabel('Mean Response (AU)')
ax.set_ylim(0, 220)
add_significance(ax, 0, 2, 200, pvalue=0.001)

# Panel E: Histogram/Distribution
ax = axes[4]
for treatment, color in zip(['Control', 'Low Dose', 'High Dose'], 
                            ['steelblue', 'coral', 'firebrick']):
    subset = treatment_df[treatment_df['treatment'] == treatment]['response']
    ax.hist(subset, bins=10, alpha=0.5, label=treatment, color=color)
ax.set_xlabel('Response (AU)')
ax.set_ylabel('Count')
ax.legend(fontsize=7, frameon=False)

# Panel F: Small heatmap
ax = axes[5]
small_heatmap = expression_df.iloc[:4, :4]
yp.heatmap(small_heatmap, cmap='RdBu_r', center=0, annot=True, fmt='.1f', ax=ax)
ax.set_title('Expression', fontsize=8)

# Add subplot labels
yplot.add_subplot_labels(fig, axes)

plt.show()

## Saving Figures for Publication

Best practices for saving figures for journals.

In [ ]:
# Create a simple figure to demonstrate saving
layout = yplot.SubplotLayout(config={
    "fig_size": (3.5, 3),  # Single column width for most journals
    "margins": {"left": 0.5, "right": 0.15, "top": 0.25, "bottom": 0.45},
    "row_1": {
        "size": (2.8, 2.2),
        "cols": 1,
        "spacing": {"hspace": 0.0, "wspace": 0.0}
    }
})

fig, axes = yplot.create_figure_with_layout(layout)

ax = axes[0]
yp.box(treatment_df, x='treatment', y='response', ax=ax)
ax.set_ylabel('Response (AU)')
ax.set_xlabel('')

plt.show()

# Saving options (uncomment to use)
# 
# For vector graphics (best for publication):
# fig.savefig('figure.pdf', dpi=300, bbox_inches='tight')
# fig.savefig('figure.svg', dpi=300, bbox_inches='tight')
# fig.savefig('figure.eps', dpi=300, bbox_inches='tight')
#
# For raster graphics:
# fig.savefig('figure.png', dpi=300, bbox_inches='tight')  # Standard
# fig.savefig('figure.png', dpi=600, bbox_inches='tight')  # High resolution
# fig.savefig('figure.tiff', dpi=300, bbox_inches='tight')  # TIFF for some journals

print("\nTypical journal figure dimensions:")
print("  - Single column: 3.5 inches (89 mm)")
print("  - 1.5 column: 5.0 inches (127 mm)")
print("  - Double column: 7.0 inches (178 mm)")
print("\nTypical DPI requirements:")
print("  - Line art: 600-1200 DPI")
print("  - Halftones/photos: 300 DPI")
print("  - Combination: 600 DPI")

## Style Presets Comparison

Comparing different style presets for different contexts.

In [ ]:
# Compare publication vs presentation styles
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

x = np.linspace(0, 10, 50)
y = np.sin(x)

# Publication style (current)
ax1.plot(x, y, 'b-', lw=1.5)
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_title('Publication Style')

# Simulate presentation style
ax2.plot(x, y, 'b-', lw=2.5)
ax2.set_xlabel('X', fontsize=12)
ax2.set_ylabel('Y', fontsize=12)
ax2.set_title('Presentation Style (larger fonts)', fontsize=14)
ax2.tick_params(labelsize=10)

plt.tight_layout()
plt.show()

print("\nStyle presets available:")
print("  yplot.use('publication')  - Small fonts, thin lines for papers")
print("  yplot.use('presentation') - Medium fonts for slides")
print("  yplot.use('poster')       - Large fonts for posters")

## Summary

Key takeaways for creating publication figures with yplot:

1. **Use SubplotLayout** for precise control over dimensions
2. **Set margins** appropriately for axis labels and titles
3. **Use yplot.add_subplot_labels()** for consistent panel labels
4. **Apply style presets** with yplot.use('publication')
5. **Add statistical annotations** with add_significance() and add_regression_stats()
6. **Save in vector format** (PDF, SVG) for best quality
7. **Check journal requirements** for dimensions and DPI before submission